In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [4]:
import pandas as pd
import os 
import numpy as np 
import cv2
from sklearn.model_selection import train_test_split 
from torch.utils.data import DataLoader
import torch 
from torchvision import transforms
from model.resnet50_lstm import ResNetLSTM
from data_ingestion.data_import import DeepfakeDataset


In [5]:
### NO NEED OF THIS 


# root = r"C:\Users\rohit\OneDrive\Desktop\DeepFakeDetectionSystem\faceframes"
# for label in ["real","fake"]:
#     input = os.path.join(root,label)
#     for video in os.listdir(input):
#         video_path = os.path.join(input,video)
#         for frames in os.listdir(video_path):
#             frame_path = os.path.join(video_path,frames)

#             img = cv2.imread(frame_path) 
#             if img is None:
#                 continue
            
#             img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
#             img = img.astype("float32") /255.0



In [6]:
data = r"C:\Users\rohit\OneDrive\Desktop\DeepFakeDetectionSystem\data.csv"
df = pd.read_csv(data)

In [7]:
df.head()

,orignal_video_path,fake_video_path,original_label,fake_label,pair_id
0,faceframes\real\video1.mp4,faceframes\fake\fakevideo1.mp4,1,0,video1
1,faceframes\real\video10.mp4,faceframes\fake\fakevideo10.mp4,1,0,video10
2,faceframes\real\video100.mp4,faceframes\fake\fakevideo100.mp4,1,0,video100
3,faceframes\real\video1000.mp4,faceframes\fake\fakevideo1000.mp4,1,0,video1000
4,faceframes\real\video101.mp4,faceframes\fake\fakevideo101.mp4,1,0,video101


In [8]:
## Data Transformation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean= [0.5,0.5,0.5],std=[0.5,0.5,0.5])
])

In [9]:
## spliting the data into train ,test and validation set

train_df,test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42
)

In [ ]:
train_dataset = DeepfakeDataset(
    df=train_df, 
    seq_len=30, 
    transform=transform
)

In [11]:
## DataLoader 

train_loader = DataLoader(
    train_dataset,
    batch_size=8,      
    shuffle=True,      
    num_workers=4,     
    pin_memory=True    
)

In [12]:
## creating the validation set 

val_df,test_df = train_test_split(
    test_df,
    test_size=0.5,
    random_state=42
)

In [13]:
print(train_df.head(2))
print(test_df.head(2))
print(val_df.head(2))

               orignal_video_path                   fake_video_path  \
541  faceframes\real\video586.mp4  faceframes\fake\fakevideo586.mp4   
440  faceframes\real\video495.mp4  faceframes\fake\fakevideo495.mp4   

     original_label  fake_label   pair_id  
541               1           0  video586  
440               1           0  video495  
               orignal_video_path                   fake_video_path  \
557   faceframes\real\video60.mp4   faceframes\fake\fakevideo60.mp4   
798  faceframes\real\video817.mp4  faceframes\fake\fakevideo817.mp4   

     original_label  fake_label   pair_id  
557               1           0   video60  
798               1           0  video817  
               orignal_video_path                   fake_video_path  \
381  faceframes\real\video441.mp4  faceframes\fake\fakevideo441.mp4   
959  faceframes\real\video962.mp4  faceframes\fake\fakevideo962.mp4   

     original_label  fake_label   pair_id  
381               1           0  video441  
959   

In [14]:
print(train_df.shape,test_df.shape,val_df.shape)

(700, 5) (150, 5) (150, 5)


In [15]:
df.head()

,orignal_video_path,fake_video_path,original_label,fake_label,pair_id
0,faceframes\real\video1.mp4,faceframes\fake\fakevideo1.mp4,1,0,video1
1,faceframes\real\video10.mp4,faceframes\fake\fakevideo10.mp4,1,0,video10
2,faceframes\real\video100.mp4,faceframes\fake\fakevideo100.mp4,1,0,video100
3,faceframes\real\video1000.mp4,faceframes\fake\fakevideo1000.mp4,1,0,video1000
4,faceframes\real\video101.mp4,faceframes\fake\fakevideo101.mp4,1,0,video101


In [16]:
import torch
import torch.nn as nn 
from torchvision import models

In [17]:
resnet = models.resnet50(pretrained = True)
resnet.fc = nn.Identity()

c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [18]:
for params in resnet.parameters():
    params.requires_grad = False

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetLSTM(resnet).to(device)


In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr =0.0001)

In [ ]:
## epochs 

model.train()
for epoch in range(50):
    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"The Epoch: {epoch+1} with loss: {loss.item():.4f}")

In [ ]:
import datetime

In [24]:
import os; print(os.getcwd())

c:\Users\rohit\OneDrive\Desktop\DeepFakeDetectionSystem\training


In [ ]:
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir ,histogram_freq=1)